In [ ]:
import os
import subprocess
import chemiscope
import ipi
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
import nqetools as nqe
from ase.visualize import view
# This follows:
# https://atomistic-cookbook.org/examples/pi-metad/pi-metad.html

In [ ]:
# Make a directory to store everything
directory_min = "min"
directory_md = "md"
directory_metamd = "meta_md"
directory_metapimd = "meta_pimd"
n_beads = 8
timestep = 1.0 # fs
total_steps = 5000
stride = 10
temperature = 298
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'zundel'
plumed_type = "mtd-coord"

In [ ]:
atoms = nqe.read_ipi_xyz("h5o2+.xyz")[-1]

In [ ]:
view(atoms)

In [ ]:
# Run minimization
atoms = nqe.run_optimise(directory_min, atoms, driver=driver_code)

In [ ]:
# Run unbiased MD
atoms = nqe.run_md(directory_md, atoms, driver=driver_code, total_steps=total_steps, temperature=temperature,
                   timestep=timestep, thermostat=thermostat, md_type=md_type, stride=stride, n_beads=1)

In [ ]:
# Run metadynamics
atoms = nqe.run_plumed_md(directory_metamd, atoms, driver=driver_code, total_steps=total_steps, temperature=temperature,
                          timestep=timestep, thermostat=thermostat, md_type=md_type, stride=stride, n_beads=1,
                          plumed_type=plumed_type)

In [ ]:
view(atoms)

In [ ]:
# Get the output data
output_data, output_desc = ipi.read_output(os.path.join(directory_metamd, "md.out"))
colvar_data = ipi.read_trajectory(os.path.join(directory_metamd, "md.colvar_0"), format="extras")[
    "d,c1.lessthan,c2.lessthan,dc,mtd.bias"
]
traj_data = ipi.read_trajectory(os.path.join(directory_metamd, "md.pos_0.xyz"))

In [1]:
# Chemiscope plot
chemiscope.show(
    frames=traj_data,
    properties=dict(
        d_OO=10 * colvar_data[:, 0],  # nm to Å
        delta_coord=colvar_data[:, 1],
        bias=27.211386 * output_data["ensemble_bias"],  # Ha to eV
        time=2.4188843e-05 * output_data["time"],  # atomictime to ps
    ),  # attime to ps
    settings=chemiscope.quick_settings(
        x="d_OO", y="delta_coord", z="bias", color="time", trajectory=True
    ),
    mode="default",
)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data)

In [ ]:
# Plot the time evolution of the energy and temperature
nqe.plot_time_temperature(output_data)

In [ ]:
n_bins = 100
cv_limits = [[0.21, 0.31], [-1, 1]]
nqe.run_plumed_hills(directory_metamd,bins=n_bins,cv=cv_limits)

In [ ]:
files = nqe.search_fes_files(directory_metamd)
# rearrange data and converts to Å and eV
data = np.loadtxt(os.path.join(directory_metamd, "FES3.dat"), comments="#")[:, :3]
xyz_0 = np.array([1, 1, 1])[:, np.newaxis, np.newaxis] * data.T.reshape(3, 101, 101)
data = np.loadtxt(os.path.join(directory_metamd, "FES4.dat"), comments="#")[:, :3]
xyz_1 = np.array([1, 1, 1])[:, np.newaxis, np.newaxis] * data.T.reshape(3, 101, 101)
data = np.loadtxt(os.path.join(directory_metamd, "FES5.dat"), comments="#")[:, :3]
xyz_2 = np.array([1, 1, 1])[:, np.newaxis, np.newaxis] * data.T.reshape(3, 101, 101)


In [ ]:
nqe.plot_energy_contour_series(xyz_0, xyz_1, xyz_2)